In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [2]:
data = pd.read_csv("../data/processed/Cleaned_Online_Retail.csv")

In [3]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Iscancelled,TransactionRevenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,20.34


In [4]:
data.describe()

,InvoiceNo,Quantity,UnitPrice,CustomerID,TransactionRevenue
count,397884.000000,397884.000000,397884.000000,397884.000000,397884.000000
mean,560616.934451,12.988238,3.116488,15294.423453,22.397000
std,13106.117773,179.331775,22.097877,1713.141560,309.071041
min,536365.000000,1.000000,0.001000,12346.000000,0.001000
25%,549234.000000,2.000000,1.250000,13969.000000,4.680000
50%,561893.000000,6.000000,1.950000,15159.000000,11.800000
75%,572090.000000,12.000000,3.750000,16795.000000,19.800000
max,581587.000000,80995.000000,8142.750000,18287.000000,168469.600000


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397884 entries, 0 to 397883
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   InvoiceNo           397884 non-null  int64  
 1   StockCode           397884 non-null  object 
 2   Description         397884 non-null  object 
 3   Quantity            397884 non-null  int64  
 4   InvoiceDate         397884 non-null  object 
 5   UnitPrice           397884 non-null  float64
 6   CustomerID          397884 non-null  int64  
 7   Country             397884 non-null  object 
 8   Iscancelled         397884 non-null  bool   
 9   TransactionRevenue  397884 non-null  float64
dtypes: bool(1), float64(2), int64(3), object(4)
memory usage: 27.7+ MB


In [6]:
data["InvoiceDate"].min()

'2010-12-01 08:26:00'

In [8]:
data["InvoiceDate"].max()

'2011-12-09 12:50:00'

In [13]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

In [14]:
# Created cutoff date , as separation between historical and future data
cutoff_date = pd.Timestamp("2011-06-01")

In [15]:
historical_df = data[data["InvoiceDate"] < cutoff_date].copy()


In [16]:
future_df = data[data["InvoiceDate"] > cutoff_date].copy()

In [18]:
historical_df.max()

InvoiceNo                                       555147
StockCode                                         POST
Description           ZINC WIRE SWEETHEART LETTER TRAY
Quantity                                         74215
InvoiceDate                        2011-05-31 15:41:00
UnitPrice                                      8142.75
CustomerID                                       18287
Country                                    Unspecified
Iscancelled                                      False
TransactionRevenue                             77183.6
dtype: object

In [19]:
# Last purchase of each customer
historical_df.groupby("CustomerID")["InvoiceDate"].max()

CustomerID
12346   2011-01-18 10:01:00
12347   2011-04-07 10:43:00
12348   2011-04-05 10:47:00
12350   2011-02-02 16:01:00
12352   2011-03-22 16:08:00
                ...        
18272   2011-04-28 18:11:00
18273   2011-03-27 11:22:00
18280   2011-03-07 09:52:00
18283   2011-05-23 11:33:00
18287   2011-05-22 10:39:00
Name: InvoiceDate, Length: 2718, dtype: datetime64[ns]

In [23]:
# Create an array for last purchase 
last_purchase =( historical_df.groupby("CustomerID")["InvoiceDate"].max() )

In [24]:
last_purchase

CustomerID
12346   2011-01-18 10:01:00
12347   2011-04-07 10:43:00
12348   2011-04-05 10:47:00
12350   2011-02-02 16:01:00
12352   2011-03-22 16:08:00
                ...        
18272   2011-04-28 18:11:00
18273   2011-03-27 11:22:00
18280   2011-03-07 09:52:00
18283   2011-05-23 11:33:00
18287   2011-05-22 10:39:00
Name: InvoiceDate, Length: 2718, dtype: datetime64[ns]

In [25]:
recency = ( cutoff_date - last_purchase ).dt.days

In [27]:
recency

CustomerID
12346    133
12347     54
12348     56
12350    118
12352     70
        ... 
18272     33
18273     65
18280     85
18283      8
18287      9
Name: InvoiceDate, Length: 2718, dtype: int64

In [29]:
recency = recency.rename("recency_days")

In [30]:
# No of unique invoices 
frequency  = ( historical_df.groupby("CustomerID")["InvoiceNo"].nunique().rename("purchase_frequency"))

In [31]:
# each customers revenue 
monetary = (
    historical_df
    .groupby("CustomerID")["TransactionRevenue"]
    .sum()
    .rename("historical_revenue")
)

In [32]:
# Total Quantity purchase
total_quantity = (
    historical_df
    .groupby("CustomerID")["Quantity"]
    .sum()
    .rename("total_quantity")
)

In [33]:
unique_products = (
    historical_df
    .groupby("CustomerID")["StockCode"]
    .nunique()
    .rename("unique_products")
)

In [34]:
# Create an array for first purchase 
first_purchase =( historical_df.groupby("CustomerID")["InvoiceDate"].min() )

In [35]:
tenure = (cutoff_date - first_purchase).dt.days 
tenure = tenure.rename("customer_tenure_days")

In [36]:
customer_features = pd.concat(
    [
        recency,
        frequency,
        monetary,
        total_quantity,
        unique_products,
        tenure
    ],
    axis=1
)

In [37]:
customer_features.shape

(2718, 6)

In [38]:
customer_features["average_order_value"] = (
    customer_features["historical_revenue"]
    / customer_features["purchase_frequency"]
)

In [39]:
customer_features["avg_quantity_per_order"] = (
    customer_features["total_quantity"]
    / customer_features["purchase_frequency"]
)

In [40]:
customer_features["orders_per_month"] = (
    customer_features["purchase_frequency"]
    / (customer_features["customer_tenure_days"] / 30 + 1)
)

In [42]:
recent_90d = historical_df[
    historical_df["InvoiceDate"] >= cutoff_date - pd.Timedelta(days=90)
]

In [43]:
revenue_90d = (
    recent_90d
    .groupby("CustomerID")["TransactionRevenue"]
    .sum()
    .rename("revenue_90d")
)

In [45]:
customer_features = customer_features.join(
    revenue_90d,
    how="left"
)

In [46]:
customer_features["revenue_90d"] = (
    customer_features["revenue_90d"]
    .fillna(0)
)

In [48]:
recent_30d = historical_df[
    historical_df["InvoiceDate"] >= cutoff_date - pd.Timedelta(days=30)
]

revenue_30d = (
    recent_30d
    .groupby("CustomerID")["TransactionRevenue"]
    .sum()
    .rename("revenue_30d")
)

In [49]:
customer_features = customer_features.join(
    revenue_30d,
    how="left"
).fillna({"revenue_30d": 0})

In [50]:
previous_90d = historical_df[
    (historical_df["InvoiceDate"] >= cutoff_date - pd.Timedelta(days=180))
    &
    (historical_df["InvoiceDate"] < cutoff_date - pd.Timedelta(days=90))
]

In [52]:
previous_revenue = (
    previous_90d
    .groupby("CustomerID")["TransactionRevenue"]
    .sum()
    .rename("previous_90d_revenue")
)

In [54]:
customer_features = customer_features.join(
    previous_revenue,
    how="left"
)

In [55]:
customer_features["previous_90d_revenue"] = (
    customer_features["previous_90d_revenue"]
    .fillna(0)
)

In [56]:
customer_features["spending_trend"] = (
    customer_features["revenue_90d"]
    - customer_features["previous_90d_revenue"]
)

In [57]:
future_revenue = (
    future_df
    .groupby("CustomerID")["TransactionRevenue"]
    .sum()
    .rename("future_revenue")
)

In [58]:
customer_features = customer_features.join(
    future_revenue,
    how="left"
)

In [59]:
customer_features["future_revenue"] = (
    customer_features["future_revenue"]
    .fillna(0)
)

In [60]:
customer_country = (
    historical_df
    .groupby("CustomerID")["Country"]
    .agg(lambda x: x.mode()[0])
    .rename("country")
)

In [61]:
customer_features = customer_features.join(
    customer_country
)

In [62]:
customer_features.shape

(2718, 15)

In [63]:
customer_features.head()

,recency_days,purchase_frequency,historical_revenue,total_quantity,unique_products,customer_tenure_days,average_order_value,avg_quantity_per_order,orders_per_month,revenue_90d,revenue_30d,previous_90d_revenue,spending_trend,future_revenue,country
CustomerID,,,,,,,,,,,,,,,
12346,133,1,77183.60,74215,1,133,77183.600000,74215.000000,0.184049,0.00,0.0,77183.60,-77183.60,0.00,United Kingdom
12347,54,3,1823.43,1117,63,175,607.810000,372.333333,0.439024,636.25,0.0,1187.18,-550.93,2486.57,Iceland
12348,56,3,1487.24,2124,22,166,495.746667,708.000000,0.459184,367.00,0.0,1120.24,-753.24,310.00,Finland
12350,118,1,334.40,197,17,118,334.400000,197.000000,0.202703,0.00,0.0,334.40,-334.40,0.00,Norway
12352,70,5,1561.81,254,26,104,312.362000,50.800000,1.119403,280.66,0.0,1281.15,-1000.49,944.23,Norway


In [64]:
customer_features.to_csv(
    "../data/processed/customer_features.csv",
    index=True
)